# M2 - Multimodal RAG Embedding Extraction (REFINED for sample_articles.csv)

This notebook rebuilds the FAISS index using **only the article IDs that exist in your `sample_articles.csv`** (41,795 articles) instead of a random sample.

## Setup Steps:
1. Create a **New Notebook** on Kaggle.
2. Add the **H&M Personalized Fashion Recommendations** dataset to the notebook.
3. Go to Session Options -> Accelerator -> **Turn on GPU T4x2**.
4. Upload this file via File -> Import Notebook.
5. In **Step 2 below**, paste your `sample_articles.csv` article IDs (copy from the cell instructions).
6. Run All cells.
7. Download `m2_clip_faiss.bin` and `m2_faiss_mapping.csv` from the **Output** tab.

## 0. Install Dependencies
Run this cell FIRST and wait for it to complete.

In [1]:
!pip install -q faiss-cpu open_clip_torch
print('Installation complete!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
Installation complete!


## 1. Import Libraries & Setup CLIP Model

In [3]:
import os
import pandas as pd
import torch
from PIL import Image
import open_clip
import faiss
import numpy as np
from tqdm.auto import tqdm

DATA_DIR = '../input/competitions/h-and-m-personalized-fashion-recommendations'

# Load CLIP Model (MUST match local app: ViT-B-32 + laion2b_s34b_b79k)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()
print('CLIP model loaded successfully!')

Using device: cuda
CLIP model loaded successfully!


## 2. Load YOUR sample_articles.csv Article IDs

**IMPORTANT:** You need to upload your `sample_articles.csv` to Kaggle.

**Option A (Recommended):** Add it as a Kaggle dataset:
- Go to kaggle.com -> Datasets -> New Dataset
- Upload your `sample_articles.csv` file
- Add that dataset to this notebook (via + Add Data)
- Update `SAMPLE_ARTICLES_PATH` below to the correct path

**Option B:** Manually paste IDs (see fallback cell below)

In [4]:
SAMPLE_ARTICLES_PATH = '/kaggle/input/datasets/maleeshavidurath/sample/sample_articles.csv'

sample_df = pd.read_csv(SAMPLE_ARTICLES_PATH)
# Normalize article_id to 10-digit zero-padded string (same as image filenames)
sample_article_ids = set(str(aid).zfill(10) for aid in sample_df['article_id'].astype(str))

print(f'Your sample_articles.csv has {len(sample_article_ids):,} unique article IDs')
print(f'Sample IDs (first 5): {list(sample_article_ids)[:5]}')

Your sample_articles.csv has 41,794 unique article IDs
Sample IDs (first 5): ['0757347001', '0708379003', '0617524001', '0867948001', '0638635002']


## 3. Filter H&M Dataset to Your Article IDs & Validate Images

In [5]:
# Load full Kaggle articles.csv
kaggle_articles_df = pd.read_csv(f'{DATA_DIR}/articles.csv')
kaggle_articles_df['article_id_str'] = kaggle_articles_df['article_id'].astype(str).str.zfill(10)
print(f'Full Kaggle dataset: {len(kaggle_articles_df):,} articles')

# Filter to only YOUR sample_articles.csv IDs
filtered_df = kaggle_articles_df[kaggle_articles_df['article_id_str'].isin(sample_article_ids)]
print(f'Articles in YOUR dataset found in Kaggle: {len(filtered_df):,}')
print(f'Articles NOT found in Kaggle (no image possible): {len(sample_article_ids) - len(filtered_df):,}')

# Validate image files exist
article_ids = []
image_paths = []

print('\nValidating image paths...')
for _, row in tqdm(filtered_df.iterrows(), total=len(filtered_df)):
    article_id = row['article_id_str']
    folder = article_id[:3]
    path = f'{DATA_DIR}/images/{folder}/{article_id}.jpg'
    if os.path.exists(path):
        article_ids.append(article_id)
        image_paths.append(path)

print(f'\nValid images found: {len(image_paths):,} / {len(filtered_df):,}')
print(f'This is {len(image_paths)/len(sample_article_ids)*100:.1f}% of your sample_articles.csv')

Full Kaggle dataset: 105,542 articles
Articles in YOUR dataset found in Kaggle: 41,794
Articles NOT found in Kaggle (no image possible): 0

Validating image paths...


  0%|          | 0/41794 [00:00<?, ?it/s]


Valid images found: 41,688 / 41,794
This is 99.7% of your sample_articles.csv


## 4. Extract CLIP Embeddings
This will take ~10-20 minutes on GPU T4x2 for ~40K images.

In [6]:
BATCH_SIZE = 256
all_embeddings = []
failed_indices = []

print(f'Running CLIP embeddings on {len(image_paths):,} images...')
with torch.no_grad():
    for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch_paths = image_paths[i:i + BATCH_SIZE]
        batch_images = []
        batch_valid_indices = []
        
        for j, path in enumerate(batch_paths):
            try:
                img = Image.open(path).convert('RGB')
                batch_images.append(preprocess(img))
                batch_valid_indices.append(i + j)
            except Exception as e:
                failed_indices.append(i + j)
        
        if not batch_images:
            continue
            
        image_tensor = torch.stack(batch_images).to(device)
        image_features = model.encode_image(image_tensor)
        # CRITICAL: Normalize for cosine similarity (IndexFlatIP)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        all_embeddings.append(image_features.cpu().numpy())

# Remove failed indices from article_ids list
if failed_indices:
    print(f'\nFailed to process {len(failed_indices)} images, removing from index...')
    article_ids = [aid for i, aid in enumerate(article_ids) if i not in set(failed_indices)]

embeddings = np.vstack(all_embeddings).astype('float32')
print(f'\nEmbeddings Shape: {embeddings.shape}')
print(f'Final article count in index: {len(article_ids):,}')

# Save intermediate results so you don't have to re-run embedding extraction
np.save('embeddings.npy', embeddings)
pd.DataFrame({'article_id': article_ids}).to_csv('article_ids_temp.csv', index=False)
print('Saved embeddings and article_ids to disk.')

Running CLIP embeddings on 41,688 images...


  0%|          | 0/163 [00:00<?, ?it/s]


Embeddings Shape: (41688, 512)
Final article count in index: 41,688
Saved embeddings and article_ids to disk.


## 5. Build FAISS Index & Export
Download both files from the **Output** tab after this cell runs.

In [7]:
# Reload if variables are lost (e.g., after kernel restart)
if 'embeddings' not in dir() or embeddings is None:
    embeddings = np.load('embeddings.npy')
    article_ids = pd.read_csv('article_ids_temp.csv')['article_id'].astype(str).tolist()
    print(f'Reloaded: {embeddings.shape}, {len(article_ids)} articles')
    
D = embeddings.shape[1]  # 512 dimensions for ViT-B-32
index = faiss.IndexFlatIP(D)  # Inner Product = Cosine Similarity for normalized vectors
index.add(embeddings)

print(f'FAISS Index built with {index.ntotal:,} vectors')

# Save Index
faiss.write_index(index, 'm2_clip_faiss.bin')

# Save mapping: FAISS index position -> article_id
pd.DataFrame({'article_id': article_ids}).to_csv('m2_faiss_mapping.csv', index=False)

# Show file sizes
faiss_size = os.path.getsize('m2_clip_faiss.bin') / (1024*1024)
mapping_size = os.path.getsize('m2_faiss_mapping.csv') / 1024
print(f'\nm2_clip_faiss.bin  : {faiss_size:.1f} MB')
print(f'm2_faiss_mapping.csv: {mapping_size:.1f} KB')
print(f'\nCoverage: {len(article_ids):,} / {len(sample_article_ids):,} articles from sample_articles.csv ({len(article_ids)/len(sample_article_ids)*100:.1f}%)')
print('\nSuccessfully Exported! Download both files from the Output folder on the right panel.')

FAISS Index built with 41,688 vectors

m2_clip_faiss.bin  : 81.4 MB
m2_faiss_mapping.csv: 447.8 KB

Coverage: 41,688 / 41,794 articles from sample_articles.csv (99.7%)

Successfully Exported! Download both files from the Output folder on the right panel.


## 6. Quick Sanity Check (Optional)

In [10]:
# Reload and verify
loaded_index = faiss.read_index('m2_clip_faiss.bin')
loaded_mapping = pd.read_csv('m2_faiss_mapping.csv')

print(f'Verified FAISS index: {loaded_index.ntotal:,} vectors')
print(f'Verified mapping CSV: {len(loaded_mapping):,} rows')
assert loaded_index.ntotal == len(loaded_mapping), 'MISMATCH! Index and mapping sizes differ!'
print('\nSanity check PASSED: index and mapping are aligned.')

# Quick search test
test_vec = embeddings[0:1]  # Use first embedding as a query
D_test, I_test = loaded_index.search(test_vec, 5)
print(f'\nTest search results (top-5 similar to article {article_ids[0]}):')
for rank, (dist, idx) in enumerate(zip(D_test[0], I_test[0])):
    print(f'  Rank {rank+1}: article_id={loaded_mapping.iloc[idx]["article_id"]}, score={dist:.4f}')

print('\n--- Zero-padding check ---')
print(loaded_mapping['article_id'].dtype)
print(loaded_mapping['article_id'].head())

Verified FAISS index: 41,688 vectors
Verified mapping CSV: 41,688 rows

Sanity check PASSED: index and mapping are aligned.

Test search results (top-5 similar to article 0108775015):
  Rank 1: article_id=108775015, score=1.0000
  Rank 2: article_id=687753001, score=0.9482
  Rank 3: article_id=623522001, score=0.9431
  Rank 4: article_id=767834001, score=0.9351
  Rank 5: article_id=779554002, score=0.9282

--- Zero-padding check ---
int64
0    108775015
1    108775044
2    110065001
3    110065002
4    110065011
Name: article_id, dtype: int64
